# Playground

In [ ]:
%matplotlib inline

import os
import sys
import glob

import matplotlib.pyplot as plt
import numpy as np

import blib
blib.useTheme("dark")

srcDir = os.path.abspath("../src")
if os.path.exists(srcDir):
    sys.path.insert(0, srcDir)

import radarkit
# import radarkit.chart

zmap = blib.matplotlibColormap("z")
vmap = blib.matplotlibColormap("v")

In [ ]:
# Create a symbolic link to the data directory
files = sorted(glob.glob(os.path.expanduser("~/Downloads/data/PX-*.rkr")))
# files = sorted(glob.glob(os.path.expanduser("~/Downloads/data/horus.rkr")))
assert len(files) > 0, "No files found"
file = files[3]
print(f"Selected file {file}")

In [ ]:
# rkid = radarkit.open(file, verbose=1)
rkid = radarkit.open(file)

config = rkid.get_current_config()
config.SQIThreshold = 0.01
config.SNRThreshold = 0.0

rkid.set_moment_method("pp")

In [ ]:
out = rkid.read()
# out = rkid.read(count=1000) # No moment can be retrieved with this

# A-Scope

- Compute the average power of the last `count` raw pulses
- Compute the average power of the last `count` compressed pulses
- Add a `tiny` value to avoid log of 0

In [ ]:
tiny = 1.0e-3

riq = out["riq"]
ciq = out["ciq"]

count = 80

plt.figure(figsize=(8, 5), dpi=200)

for k in range(2):
    r = np.mean(np.abs(riq[-count:, k, :]) ** 2, axis=0) if riq is not None else np.zeros(ciq.shape[2])
    c = np.mean(np.abs(ciq[-count:, k, :]) ** 2, axis=0)

    xr = np.arange(riq.shape[2]) if riq is not None else np.arange(ciq.shape[2])
    xc = np.arange(ciq.shape[2]) * rkid.desc.pulseToRayRatio
    xb = np.array([1, 1]) * xc[-1]

    yr = 10 * np.log10(r + tiny)
    yc = 10 * np.log10(c + tiny)

    ax = plt.subplot(2, 1, k + 1)
    plt.plot(xr, yr, label="raw")
    plt.plot(xc, yc, label="compressed")
    plt.plot(xb, [0, 100], ":")
    plt.ylim(0.5, 90)
    plt.grid()
    if k == 0:
        plt.title(f"A-Scope -- {os.path.basename(file)}", fontweight="bold")
        ax.set_xticklabels([])
        plt.text(
            xb[0] + 50,
            65,
            "End Blind Zone",
            bbox={"boxstyle": "round, pad=0.5", "facecolor": (0.25, 0.25, 0.35, 0.9), "alpha": 0.8},
        )
    else:
        plt.legend(ncols=2)
        plt.xlabel("Range Gates")
    plt.ylabel("Power (dB-ADU)")

# B-Scope

- Select a `count` for number of pulses to compute a radial
- Extract up to `rayCount * count` pulses
- Reindex the array to (gateCount, rayCount, count)
- Compute power, average over the last axis count

In [ ]:
count = 80
rayCount = ciq.shape[0] // count
gateCount = ciq.shape[2]
n = rayCount * count
x = np.reshape(ciq[:n, 0, :], (rayCount, count, -1))
x = np.transpose(x, (2, 0, 1))

r0 = np.mean(np.abs(x) ** 2, axis=2)
r1 = np.mean(x[:, :, 1:] * np.conj(x[:, :, :-1]), axis=2)
r2 = np.mean(x[:, :, 2:] * np.conj(x[:, :, :-2]), axis=2)

In [ ]:
plt.figure(figsize=(8, 4), dpi=200)
plt.pcolormesh(10 * np.log10(r0 + tiny), cmap="viridis", vmin=8, vmax=42)
plt.xlabel("Ray Index")
plt.ylabel("Gate Index")
plt.title("B-Scope", fontweight="bold")
cb = plt.colorbar()
cb.set_label("dB-ADU")

In [ ]:
plt.figure(figsize=(8, 4), dpi=200)
# plt.pcolormesh(np.abs(r1) / (np.abs(r0) + tiny), cmap='viridis')
# plt.pcolormesh(10 * np.log10(np.abs(r1) + tiny), cmap='viridis', vmin=8, vmax=42)
plt.pcolormesh(np.angle(r1), cmap=vmap, vmin=-np.pi, vmax=np.pi)
# plt.pcolormesh(np.log(np.abs(r1) / np.abs(r2) + tiny), cmap='tab20c')
plt.xlabel("Ray Index")
plt.ylabel("Gate Index")
plt.title("B-Scope", fontweight="bold")
cb = plt.colorbar()
cb.set_label("dB-ADU")

## Check the Next Pulse Status

For debuggning purpose, the next pulse header status should be 0

In [ ]:
k = rkid.pulseMachine.contents.pulseIndex.contents.value
pulse = radarkit.RKGetPulseFromBuffer(rkid.pulseMachine.contents.pulseBuffer, k)
print(f"pulses[{k}].header.s = {pulse.contents.header.s}")

## Retrieve Radar Products

In [ ]:
assert rkid.sweepMachine.contents.lastRecordedScratchSpaceIndex >= 0, "Incomplete Sweep Data"

In [ ]:
# rkid.flush()
prods = rkid.get_moment()

In [ ]:
z = np.transpose(prods["Z"])

plt.figure(figsize=(8, 4), dpi=200)
# plt.pcolormesh(z, cmap='viridis', vmin=2, vmax=42)
plt.pcolormesh(z, cmap=zmap, vmin=-32, vmax=96)
plt.xlabel("Ray Index")
plt.ylabel("Gate Index")
plt.title("B-Scope", fontweight="bold")
cb = plt.colorbar()
cb.set_label("dBZ")

In [ ]:
rkid.free()
plt.close()

In [ ]:
sweep = {
    "kind": "Plain",
    "txrx": "M",
    "time": prods["time"],
    "latitude": rkid.desc.latitude,
    "longitude": rkid.desc.longitude,
    "sweepMode": "ppi",
    "sweepElevation": rkid.configs[0].sweepElevation,
    "sweepAzimuth": rkid.configs[0].sweepAzimuth,
    "prf": rkid.configs[0].prt[0] ** -1,
    "waveform": rkid.configs[0].waveform.contents.name.decode("utf"),
    "gatewidth": rkid.configs[0].pulseGateSize * rkid.desc.pulseToRayRatio,
    "elevations": prods["elevations"],
    "azimuths": prods["azimuths"],
    "ranges": prods["ranges"],
    "products": {
        "Z": prods["Z"],
        "V": prods["V"],
        "W": prods["W"],
        "D": prods["D"],
        "P": prods["P"],
        "R": prods["R"],
    },
}

In [ ]:
import radar
import radar.chart

In [ ]:
chart = radar.chart.ChartSinglePPI(sweep)

In [ ]:
chart.fig.savefig(os.path.expanduser("~/Downloads/ppi-gmap.png"))